# Convolución y atención, mirando la misma imagen

_Qué supone cada arquitectura y cómo se le nota_

Este cuaderno acompaña al capítulo [De la convolución a la atención](https://iraitzm.github.io/manual-ia-generativa/parts/fundamentos/redes.html).

La idea es sencilla: coger una imagen, pasarla por una red convolucional y por un transformer de visión, y **provocar que se equivoquen de maneras distintas**. Que se equivoquen distinto es lo interesante, porque el modo de fallar de un modelo se deduce de lo que ese modelo da por supuesto.

Es el ejercicio que propuso [Julia Turc](https://www.youtube.com/watch?v=KnCRTP11p5U) y consiste en enseñarles un gato con plumaje de periquito: la forma dice una cosa y la textura dice otra.

Ninguna sección necesita GPU ni credenciales. La última, opcional, sí pide una cuenta de Hugging Face para generar variantes de la imagen.

## Preparación

In [ ]:
%pip install -q "transformers>=4.44" torch scikit-image scipy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

SEMILLA = 42
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

Empezamos con una gata corriente, la que trae `scikit-image`, que hará de control: es la imagen sobre la que los dos modelos deberían coincidir.

In [ ]:
from skimage import data

gato = data.chelsea()  # una gata llamada Chelsea, 300x451 píxeles
print(gato.shape, gato.dtype)

plt.figure(figsize=(5, 4))
plt.imshow(gato)
plt.axis("off")
plt.show()

## Qué hace realmente un filtro

Antes de cargar nada grande, conviene ver la operación que da nombre a las redes convolucionales. Es una ventanita de números que se pasea por la imagen multiplicando y sumando.

In [ ]:
from scipy.signal import convolve2d

gris = gato.mean(axis=2)

FILTROS = {
    "bordes verticales": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "bordes horizontales": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "desenfoque": np.ones((3, 3)) / 9,
}

fig, ejes = plt.subplots(1, 4, figsize=(16, 4))
ejes[0].imshow(gris, cmap="gray")
ejes[0].set_title("original")

for eje, (nombre, filtro) in zip(ejes[1:], FILTROS.items()):
    eje.imshow(convolve2d(gris, filtro, mode="same"), cmap="gray")
    eje.set_title(f"{nombre}\n{filtro.size} pesos")

for eje in ejes:
    eje.axis("off")
plt.tight_layout()
plt.show()

Nueve números detectan bordes verticales **en cualquier punto de la imagen**. Ahí están las dos suposiciones de la convolución hechas código: que lo importante de un píxel está en sus vecinos (la ventana es de 3 por 3) y que un borde es un borde en cualquier sitio (el mismo filtro se aplica a todo).

En una red no se escriben a mano: se aprenden. Lo que no se aprende es el tamaño de la ventana, y de ahí viene la limitación que interesa.

### El campo receptivo

Cada capa amplía un poco el trozo de imagen que influye en una neurona. Poco.

In [ ]:
def campo_receptivo(capas, ventana=3):
    """Píxeles de lado que ve una neurona tras apilar capas de convolución."""
    return 1 + capas * (ventana - 1)

for capas in [1, 2, 5, 10, 20, 50]:
    lado = campo_receptivo(capas)
    print(f"{capas:3} capas → ve {lado:3}x{lado:<3} píxeles  "
          f"({100 * lado / 451:.0f}% del ancho de la imagen)")

Cincuenta capas para abarcar la imagen entera. Un transformer, en cambio, relaciona cualquier posición con cualquier otra **en la primera capa**, porque su operación no tiene ventana. Esa es toda la diferencia, y el resto del cuaderno consiste en verla en los resultados.

## Los dos modelos

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODELOS = {
    "ResNet-50 (convolucional)": "microsoft/resnet-50",
    "ViT-base (transformer)": "google/vit-base-patch16-224",
}

cargados = {}
for etiqueta, nombre in MODELOS.items():
    # `eager` solo aplica al transformer: es lo que permite recuperar los pesos
    # de atención más adelante. Una convolucional no tiene atención que sacar.
    extra = {"attn_implementation": "eager"} if "vit" in nombre else {}
    cargados[etiqueta] = (
        AutoImageProcessor.from_pretrained(nombre),
        AutoModelForImageClassification.from_pretrained(nombre, **extra),
    )
    print(f"{etiqueta}: cargado")

Los dos están entrenados sobre ImageNet, con las mismas mil categorías. Comparten datos y tarea, así que lo único que los separa es la arquitectura.

In [ ]:
def predecir(imagen, k=5):
    """Las k categorías más probables según cada modelo."""
    salida = {}
    for etiqueta, (procesador, modelo) in cargados.items():
        entradas = procesador(imagen, return_tensors="pt")
        with torch.no_grad():
            logits = modelo(**entradas).logits
        probabilidades = logits.softmax(-1)[0]
        mejores = probabilidades.topk(k)
        salida[etiqueta] = [
            (modelo.config.id2label[i.item()], round(p.item(), 3))
            for p, i in zip(mejores.values, mejores.indices)
        ]
    return salida

def comparar(imagen, titulo):
    print(f"\n=== {titulo} ===")
    for etiqueta, top in predecir(imagen).items():
        print(f"\n{etiqueta}")
        for nombre, probabilidad in top:
            print(f"   {probabilidad:.3f}  {nombre}")

comparar(Image.fromarray(gato), "La gata de control")

Con una foto normal los dos aciertan y no hay nada que discutir. La gracia empieza al romper la imagen de una forma concreta.

## El experimento: separar forma de textura

Una red convolucional construye su representación apilando texturas locales. Un transformer puede juzgar por la silueta desde el principio. Si esa descripción es correcta, **una imagen con la forma de una cosa y la textura de otra debería separarlos**.

La imagen la generó un modelo de difusión pidiéndole un gato con plumaje de periquito, y sale bastante bien parada para lo que necesitamos: la silueta, la cara y las patas son inequívocamente de gato, mientras que el cuerpo tiene plumas de colores saturados y hasta un ala. Vive en el repositorio del manual, así que el cuaderno se la descarga.

In [ ]:
import urllib.request
from pathlib import Path

URL = "https://raw.githubusercontent.com/IraitzM/manual-ia-generativa/main/images/carrot.png"

ruta = Path("carrot.png")
if not ruta.exists():
    en_clon = Path("../../images/carrot.png")   # si se ejecuta desde el repositorio
    if en_clon.exists():
        ruta = en_clon
    else:
        urllib.request.urlretrieve(URL, ruta)

periquito = Image.open(ruta).convert("RGB")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].imshow(gato)
ejes[0].set_title("control: forma y textura de gato")
ejes[1].imshow(periquito)
ejes[1].set_title("forma de gato, textura de periquito")
for eje in ejes:
    eje.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
comparar(periquito, "El gato con plumaje de periquito")

> **Qué mirar, y qué no dar por hecho**
>
> No miréis solo la primera etiqueta. Mirad **cuánta probabilidad se lleva** y qué hay en las otras cuatro: ahí es donde se ve si el modelo dudaba entre un felino y un ave, o si ni se lo planteó.
>
> Lo que la literatura documenta es una tendencia, no una ley: las convolucionales entrenadas sobre ImageNet [tienden a decidir por textura más que por forma](https://arxiv.org/abs/1811.12231). Con esta imagen y estos dos modelos puede salir claro, puede salir tibio o pueden coincidir los dos. Si coinciden, eso también es un resultado, y el ejercicio 1 propone cómo forzar la separación.
>
> Y un aviso sobre las etiquetas: ImageNet no tiene una categoría "gato" ni una categoría "loro". Tiene razas concretas (`tabby`, `Egyptian cat`) y especies concretas (`macaw`, `lorikeet`), así que lo que hay que leer es la familia a la que pertenece la etiqueta, no la etiqueta.


### Ejercicio 1

Fabricad una versión **más extrema** de la misma idea: quedaos con la silueta de la gata de control, borrando el pelo con un desenfoque, y pintadle encima una textura de colores. Es una quimera peor que la generada, pero tiene una ventaja grande, que se puede graduar.

La celda siguiente lo monta. Variad `sigma` y anotad cómo cambia la probabilidad de la categoría felina en cada modelo: con `sigma` bajo la textura de pelo sigue ahí y los dos deberían acertar; según sube, la única pista que queda es el contorno. **El modelo que aguante más es el que menos depende de la textura.**

In [ ]:
from scipy.ndimage import gaussian_filter

alto, ancho = gris.shape
yy, xx = np.mgrid[0:alto, 0:ancho]

# Un patrón periódico de colores saturados haciendo de plumaje
plumaje = np.stack([
    0.5 + 0.5 * np.sin(xx / 5.0 + yy / 17.0),
    0.5 + 0.5 * np.sin(yy / 4.0),
    0.5 + 0.5 * np.sin((xx + yy) / 6.0),
], axis=-1)


def quimera(sigma):
    """Silueta de la gata (frecuencias bajas) con el plumaje encima."""
    silueta = gaussian_filter(gris, sigma=sigma) / 255.0 if sigma > 0 else gris / 255.0
    mezcla = np.clip(plumaje * silueta[..., None] * 1.6, 0, 1)
    return Image.fromarray((mezcla * 255).astype("uint8"))


fig, ejes = plt.subplots(1, 4, figsize=(16, 4))
for eje, sigma in zip(ejes, [0, 2, 4, 10]):
    eje.imshow(quimera(sigma))
    eje.set_title(f"sigma = {sigma}")
    eje.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
for sigma in [0, 4, 10]:
    comparar(quimera(sigma), f"Quimera sintética, sigma = {sigma}")

### Ejercicio 2

Probad otras texturas sobre la misma silueta: rayas anchas de cebra, lunares, ruido de colores, un degradado suave. Y probad también lo contrario, la textura de la gata sobre la silueta de otra cosa.

La pregunta a responder es cuál de los dos modelos cambia más de opinión, y si esa diferencia se mantiene con todas las texturas o solo con algunas. Si solo con algunas, mirad qué tienen en común las que funcionan.

## Dónde mira un transformer

El ViT parte la imagen en cuadraditos de 16 por 16 píxeles y trata cada uno como si fuera un token. Con una entrada de 224 por 224 salen 196 trozos, más un token especial que resume la imagen entera.

Lo interesante es hacerlo sobre el gato con plumas, porque ahí la imagen tiene dos regiones que dicen cosas distintas: la cara dice gato y el cuerpo dice pájaro. El mapa de atención muestra a cuál de las dos hace caso.

In [ ]:
procesador_vit, modelo_vit = cargados["ViT-base (transformer)"]
entradas = procesador_vit(periquito, return_tensors="pt")

with torch.no_grad():
    salida = modelo_vit(**entradas, output_attentions=True)

print(f"Capas de atención: {len(salida.attentions)}")
print(f"Forma de cada una: {tuple(salida.attentions[0].shape)}  (lote, cabezas, tokens, tokens)")
print(f"Tokens: 1 de resumen + {14 * 14} trozos de 16x16 píxeles")

Esa matriz de 197 por 197 es literalmente el mecanismo de atención: cuánto mira cada trozo a cada otro trozo. La fila del token de resumen dice de qué partes de la imagen sale la decisión final.

In [ ]:
# Promedio sobre las cabezas de la última capa, fila del token de resumen
atencion = salida.attentions[-1][0].mean(0)[0, 1:].reshape(14, 14).numpy()

entrada_vista = entradas["pixel_values"][0].permute(1, 2, 0).numpy()
entrada_vista = (entrada_vista - entrada_vista.min()) / np.ptp(entrada_vista)

fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
ejes[0].imshow(entrada_vista)
ejes[0].set_title("lo que entra al modelo (224x224)")
ejes[1].imshow(atencion, cmap="inferno")
ejes[1].set_title("atención del token de resumen")
ejes[2].imshow(entrada_vista)
ejes[2].imshow(np.kron(atencion, np.ones((16, 16))), cmap="inferno", alpha=0.55)
ejes[2].set_title("superpuestas")

for eje in ejes:
    eje.axis("off")
plt.tight_layout()
plt.show()

Nadie le enseñó al modelo dónde está la cara del gato. Esos pesos salen de entrenar a clasificar, y son la respuesta visual a la pregunta de qué significa "prestar atención" en una arquitectura de este tipo.

Con esta imagen en concreto la pregunta que merece la pena hacerse es si el foco cae sobre la cara, sobre el plumaje o repartido, y si eso cuadra con la etiqueta que salió antes.

> **Un mapa de atención no es una explicación**
>
> Es tentador leer estos mapas como el motivo de la decisión. Conviene resistirse: son los pesos de la última capa promediados sobre las cabezas, y hay doce capas debajo cuyas contribuciones no se ven aquí. Distintas cabezas atienden a cosas muy distintas, y promediarlas mezcla señales que no son comparables.
>
> Sirven para intuir, no para justificar. Es la misma cautela que merece cualquier técnica de interpretabilidad.


### Ejercicio 3

Repetid el mapa con la gata de control y ponedlos uno al lado del otro. ¿Mira el modelo a la misma zona en las dos imágenes?

### Ejercicio 4

Dibujad la atención de **cada una de las doce cabezas** de la última capa por separado, en lugar del promedio, y después haced lo mismo con la primera capa.

Dos patrones que buscar: cabezas que miran a todas partes por igual y cabezas muy concentradas en una zona; y la diferencia entre las capas de abajo, que suelen mirar cerca, y las de arriba, que miran lejos. Esa progresión es lo que en una convolucional se conseguía apilando capas y aquí está disponible desde el principio.

## De dónde sale la imagen

El gato con plumas no es una foto: lo generó [FLUX.1](https://huggingface.co/black-forest-labs/FLUX.1-dev) a partir de una descripción, y está guardado en el repositorio para que el cuaderno no dependa de credenciales.

Esta celda es la que lo produjo y sirve para fabricar variantes: otro animal, otra textura, otro grado de mezcla. Hace falta una cuenta de Hugging Face con un token con permiso de inferencia, y es la única celda del cuaderno que puede costar dinero según el proveedor y el plan.

In [ ]:
from getpass import getpass
from huggingface_hub import InferenceClient

cliente = InferenceClient(api_key=getpass("Token de https://huggingface.co/settings/tokens: "))

variante = cliente.text_to_image(
    "A cat with colorful parrot-like plumage",
    model="black-forest-labs/FLUX.1-dev",
)
variante.save("variante.png")

comparar(variante, "Variante generada")

### Ejercicio 5

Generad cinco variantes con la misma descripción y clasificadlas todas. Veréis que el resultado baila bastante de una a otra, porque cada imagen reparte la forma y la textura de manera distinta: en unas el plumaje cubre la cara y en otras no.

Eso es un recordatorio útil para cualquier evaluación de modelos de visión: **una sola imagen no demuestra nada**. Lo que se mide es la tendencia sobre un conjunto, y por eso el trabajo original sobre el sesgo de textura usó miles de imágenes y no una.

## Para seguir

* [Un transformer actual, pieza a pieza](https://colab.research.google.com/github/IraitzM/manual-ia-generativa/blob/main/notebooks/modelos/qwen-desde-cero.ipynb), el cuaderno que monta las piezas internas.
* [Qué hay dentro](https://iraitzm.github.io/manual-ia-generativa/parts/modelos/transformers.html), donde la atención se cuenta aplicada a texto.
* [Los retos de seguridad](https://iraitzm.github.io/manual-ia-generativa/parts/seguridad/retos.html), porque que una imagen y un texto compartan representación tiene consecuencias que no son solo técnicas.